# Extended frozen zero-shot TVPReid benchmark (Colab)

Runs one model at a time through the repository's Stage 1 evaluator. Results are resume-safe and contain only measurements produced in this runtime. Start with `MODEL = "qwen3_vl_embed_2b"` and `SUBSETS = ["prid"]`.

There are two incompatible dependency profiles: **modern** uses Transformers 4.57.3 for OpenAI CLIP, Jina CLIP v2, and Qwen3-VL-Embedding; **gme** uses Transformers 4.51.3 for GME-Qwen2-VL. When switching profiles, the install cell stops and tells you to restart the runtime. No Hugging Face cache files are patched.

In [ ]:
# 1. Verify the Colab GPU without importing torch before the pinned install.
import subprocess

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True, check=True)
print(gpu.stdout)
if "T4" not in gpu.stdout:
    print("WARNING: this notebook is sized for a Tesla T4 16 GB.")

In [ ]:
# 2. Clone or update only the feature branch. Never reset to origin/main.
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/BASSAT-BASSAT/Benchmarking-Video-Image-language-models-for-Tracklet-retrieval-.git"
BRANCH = "feat/extended-zero-shot-benchmark"
REPO_DIR = Path("/content/shawaf-vlm")

def run(command, cwd=None):
    print("+", " ".join(map(str, command)), flush=True)
    subprocess.check_call([str(part) for part in command], cwd=str(cwd) if cwd else None)

if (REPO_DIR / ".git").is_dir():
    run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run(["git", "checkout", BRANCH], cwd=REPO_DIR)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR)
else:
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR])
print(run(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=REPO_DIR))

In [ ]:
# 3. Run configuration: exactly one model; start with PRID.
MODEL = "qwen3_vl_embed_2b"
SUBSETS = ["prid"]  # later: ["prid", "ilids", "duke"]
DEVICE = "cuda"
SPLIT = "test"
TEXT_BATCH = 8
MOUNT_DRIVE = False

SUPPORTED = {"openai_clip_vit_l14", "jina_clip_v2", "gme_qwen2_vl_2b", "qwen3_vl_embed_2b"}
PROFILE_BY_MODEL = {"openai_clip_vit_l14": "modern", "jina_clip_v2": "modern", "qwen3_vl_embed_2b": "modern", "gme_qwen2_vl_2b": "gme"}
if MODEL not in SUPPORTED:
    raise ValueError(f"Choose one extended model: {sorted(SUPPORTED)}")
if not SUBSETS or not set(SUBSETS) <= {"prid", "ilids", "duke"}:
    raise ValueError("SUBSETS must contain prid, ilids, and/or duke.")
ENV_PROFILE = PROFILE_BY_MODEL[MODEL]
MODEL_BATCH = {"openai_clip_vit_l14": 4, "jina_clip_v2": 1, "gme_qwen2_vl_2b": 1, "qwen3_vl_embed_2b": 1}
BATCH = MODEL_BATCH[MODEL]

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_DIR = Path("/content/drive/MyDrive/tvpreid_extended_results")
else:
    RESULTS_DIR = Path("/content/tvpreid_extended_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(MODEL, SUBSETS, "profile", ENV_PROFILE, "batch", BATCH, "results", RESULTS_DIR)

In [ ]:
# 4. Install the model-specific environment. Avoid Torch 2.11 + cu130.
from importlib.metadata import PackageNotFoundError, version

TARGET_TRANSFORMERS = {"modern": "4.57.3", "gme": "4.51.3"}[ENV_PROFILE]
COMMON = ["accelerate==1.12.0", "qwen-vl-utils==0.0.14", "huggingface-hub==0.36.0", "opencv-python-headless==4.12.0.88", "pandas==2.2.3", "pillow==11.3.0"]
PROFILE_PACKAGES = {
    "modern": ["transformers==4.57.3", "einops==0.8.1", "timm==1.0.20"],
    "gme": ["transformers==4.51.3", "sentence-transformers==5.1.2"],
}

def installed(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None

before_transformers = installed("transformers")
before_torch = installed("torch")
before_torchvision = installed("torchvision")
loaded_transformers = "transformers" in sys.modules
loaded_torch = "torch" in sys.modules
loaded_torchvision = "torchvision" in sys.modules
torch_stack_matches = before_torch == "2.8.0+cu126" and before_torchvision == "0.23.0+cu126"
if not torch_stack_matches:
    run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--index-url", "https://download.pytorch.org/whl/cu126", "torch==2.8.0", "torchvision==0.23.0"])
run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *COMMON, *PROFILE_PACKAGES[ENV_PROFILE]])
if ENV_PROFILE == "modern":
    run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/openai/CLIP.git"])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_DIR)])

restart_reasons = []
if loaded_transformers and before_transformers != TARGET_TRANSFORMERS:
    restart_reasons.append(f"Transformers changed from {before_transformers} to {TARGET_TRANSFORMERS}")
if (loaded_torch or loaded_torchvision) and not torch_stack_matches:
    restart_reasons.append(f"Torch stack changed from torch={before_torch}, torchvision={before_torchvision} to cu126")
if restart_reasons:
    print("\nRUNTIME RESTART REQUIRED:")
    print("; ".join(restart_reasons))
    print("Use Runtime > Restart session, then run the notebook again from cell 1 with the same MODEL.")
    raise RuntimeError("Restart the Colab runtime before importing model libraries.")

import torch, torchvision, transformers
print("profile", ENV_PROFILE, "torch", torch.__version__, "torchvision", torchvision.__version__, "cuda", torch.version.cuda, "transformers", transformers.__version__)
assert torch.cuda.is_available(), "Enable a GPU runtime before continuing."
assert torch.__version__ == "2.8.0+cu126" and torchvision.__version__ == "0.23.0+cu126" and torch.version.cuda == "12.6"
assert transformers.__version__ == TARGET_TRANSFORMERS

In [ ]:
# 5. Download only test data needed for selected subsets plus required PRID smoke validation.
from shawaf_vlm.data.tvpreid import download_tvpreid, load_tvpreid

required_subsets = tuple(dict.fromkeys(["prid", *SUBSETS]))
DATA_ROOT = download_tvpreid(configs=required_subsets, split=SPLIT, local_dir=Path("/content/TVPReid"))
FRAME_CACHE = DATA_ROOT / "frame_cache"
print("TVPReid", DATA_ROOT, "cache", FRAME_CACHE)

In [ ]:
# 6. Mandatory model contract validation before a benchmark run.
import numpy as np
from shawaf_vlm.models import build_encoder
from shawaf_vlm.sampling import resolve_frame_paths

torch.cuda.reset_peak_memory_stats()
encoder = build_encoder(MODEL, device=DEVICE)
text_probe = encoder.encode_texts(["A person walking outdoors.", "A person carrying a bag."], batch_size=2 if MODEL != "qwen3_vl_embed_2b" else 1)
assert text_probe.shape[0] == 2 and np.isfinite(text_probe).all()
np.testing.assert_allclose(np.linalg.norm(text_probe, axis=1), 1.0, atol=2e-3)
prid = load_tvpreid("prid", split=SPLIT, root=DATA_ROOT)
probe_frames = resolve_frame_paths(prid.gallery[0].crop_paths, num_frames=8, frame_cache=FRAME_CACHE, sample="uniform")
video_probe = encoder.encode_videos([probe_frames], batch_size=1)
assert video_probe.shape == (1, text_probe.shape[1]) and np.isfinite(video_probe).all()
np.testing.assert_allclose(np.linalg.norm(video_probe, axis=1), 1.0, atol=2e-3)
print("probe shapes", text_probe.shape, video_probe.shape, "cosine", float(text_probe[0] @ video_probe[0]))

In [ ]:
# 7. Resume-safe evaluation. PRID uniform8 is always the first full run.
import gc
import json
import traceback
from datetime import datetime, timezone

from shawaf_vlm.eval_loop import evaluate_text_to_tracklet, evaluate_text_to_tracklet_windows
from shawaf_vlm.metrics import format_metrics

POOLS = ("mean", "mean_s8", "max", "query_max")
WINDOW_PROTOCOLS = [
    {"name": "vt_1fps_n12", "sample_fps": 1.0, "max_frames": 12, "stride": 4},
    {"name": "vt_2fps_n32", "sample_fps": 2.0, "max_frames": 32, "stride": 4},
    {"name": "reid_8fps_n64", "sample_fps": 8.0, "max_frames": 64, "stride": 4},
]
METRIC_KEYS = ["Rank-1", "Rank-5", "Rank-10", "Rank-20", "Rank-50", "mAP", "MdR", "MnR", "nDCG@10", "mINP", "num_valid_queries", "num_gallery", "num_clips", "decode_s", "video_s", "text_s", "score_s", "total_s", "video_ms_per_item", "peak_gpu_gb", "reserved_gpu_gb"]

def result_path(subset, protocol, pool):
    return RESULTS_DIR / f"{MODEL}_tvpreid_{subset}_{protocol}_{pool}.json"

def save_result(subset, protocol, pool, sampling, metrics=None, error=None):
    row = {"model": MODEL, "subset": subset, "split": SPLIT, "protocol": protocol, "pool": pool, "sampling": sampling, "created_at": datetime.now(timezone.utc).isoformat(), "status": "failed" if error else "complete"}
    if metrics is not None:
        row.update({key: float(metrics[key]) for key in METRIC_KEYS if key in metrics})
    if error:
        row["error"] = str(error)
    path = result_path(subset, protocol, pool)
    path.write_text(json.dumps(row, indent=2), encoding="utf-8")
    print("Wrote", path, flush=True)

def complete(path):
    if not path.is_file():
        return False
    try:
        return json.loads(path.read_text(encoding="utf-8")).get("status") == "complete"
    except Exception:
        return False

def clean_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def run_uniform8(subset):
    path = result_path(subset, "uniform8", "none")
    if complete(path):
        print("Resume: skip", path.name); return True
    splits = load_tvpreid(subset, split=SPLIT, root=DATA_ROOT)
    sampling = "uniform 8 ordered frames"
    try:
        metrics = evaluate_text_to_tracklet(encoder, splits, num_frames=8, batch_size=BATCH, text_batch_size=TEXT_BATCH, junk_same_camera=False, frame_cache=FRAME_CACHE, frame_sample="uniform")
        print(format_metrics(metrics)); save_result(subset, "uniform8", "none", sampling, metrics=metrics); return True
    except RuntimeError as exc:
        save_result(subset, "uniform8", "none", sampling, error=exc)
        if "out of memory" in str(exc).lower(): clean_cuda(); return False
        raise

def run_windows(subset, spec):
    paths = [result_path(subset, spec["name"], pool) for pool in POOLS]
    if all(complete(path) for path in paths):
        print("Resume: skip", subset, spec["name"]); return True
    splits = load_tvpreid(subset, split=SPLIT, root=DATA_ROOT)
    sampling = f"ordered 8-frame clips, stride {spec['stride']}, {spec['sample_fps']:g} fps, cap {spec['max_frames']}"
    try:
        scored = evaluate_text_to_tracklet_windows(encoder, splits, num_frames=8, stride=spec["stride"], sample_fps=spec["sample_fps"], max_frames=spec["max_frames"], pools=POOLS, batch_size=BATCH, text_batch_size=TEXT_BATCH, junk_same_camera=False, frame_cache=FRAME_CACHE)
        for pool, metrics in scored.items():
            print(pool); print(format_metrics(metrics)); save_result(subset, spec["name"], pool, sampling, metrics=metrics)
        return True
    except RuntimeError as exc:
        for pool, path in zip(POOLS, paths):
            if not complete(path): save_result(subset, spec["name"], pool, sampling, error=exc)
        if "out of memory" in str(exc).lower(): clean_cuda(); return False
        raise

# Gate: PRID uniform8 must pass before any other protocol.
if not run_uniform8("prid"):
    del encoder
    clean_cuda()
    raise RuntimeError("CUDA OOM during PRID uniform8; model was released and benchmark parameters were not changed.")
for subset in SUBSETS:
    if subset != "prid" and not run_uniform8(subset): break
    for spec in WINDOW_PROTOCOLS:
        if not run_windows(subset, spec):
            print("CUDA OOM: stopping this model without changing benchmark parameters.")
            break


In [ ]:
# 8. Free GPU memory and build the final comparison dataframe from saved files.
del encoder, text_probe, video_probe
clean_cuda()

import pandas as pd
rows = []
for path in sorted(RESULTS_DIR.glob("*_tvpreid_*.json")):
    try:
        rows.append(json.loads(path.read_text(encoding="utf-8")))
    except Exception as exc:
        print("Skip unreadable result", path, exc)
comparison = pd.DataFrame(rows)
if not comparison.empty:
    comparison = comparison.sort_values(["model", "subset", "protocol", "pool"], kind="mergesort")
    comparison.to_csv(RESULTS_DIR / "extended_zero_shot_comparison.csv", index=False)
display(comparison)
print("Saved", RESULTS_DIR / "extended_zero_shot_comparison.csv")